In [ ]:
#| default_exp machine_learning.llm_typo_finding

In [ ]:
#| export

from os import PathLike
import re
from typing import List, Optional

from pydantic import BaseModel, Field, field_validator, ValidationError

from trouver.llm_core.call_llm import SupportedLLM, call_llm, process_llm_response, smart_truncate
from trouver.obsidian.vault import NoteDoesNotExistError, VaultNote, NoteNotUniqueError
from trouver.obsidian.file import MarkdownFile
from trouver.personal_vault.notes import notes_linked_in_note
from trouver.personal_vault.note_processing import process_standard_information_note

In [ ]:
#| export
class LaTeXCorrection(BaseModel):
    original: str = Field(..., description="The segment with the typo")
    correction: str = Field(..., description="The corrected LaTeX string")
    reason: str = Field(..., description="Why this was flagged")
    confidence: float = Field(..., description="Confidence score between 0.0 and 1.0", ge=0, le=1)

    @field_validator('correction')
    @classmethod
    def check_latex_syntax(cls, v: str) -> str:
        # 1. Check for unbalanced curly braces - a classic LLM error
        if v.count('{') != v.count('}'):
            raise ValueError("Unbalanced curly braces in LaTeX output.")
        
        # 2. Check for common 'double-escape' errors caused by JSON/Python strings
        # LLMs sometimes output \\\\alpha instead of \\alpha
        if "\\\\" in v and not any(x in v for x in ["\\\\", "\\newline", "\\hline"]):
            # Note: This is a soft-warning logic, can be adjusted
            pass 

        # 3. Ensure math mode symbols are likely contained in $ or \[ if they are stand-alone
        # (This is more complex, but simple regex can catch basic issues)
        return v

class TypoReport(BaseModel):
    paper_title: str
    errors: List[LaTeXCorrection]

In [ ]:
#| export
FIND_LATEX_TYPOS_SYSTEM_PROPT = rf"""
**Role**: Transcription Auditor for Mathematical Manuscripts.
**Task**: Identify corruptions introduced during OCR and literal translation, focusing on LaTeX boundary errors and terminology. You are NOT to argue with the core content or logic of the text. Your goal is to fix accidental corruptions from OCR, NOT to reformat intentional Markdown/HTML structural elements provided in the text.

### I. Principles of Boundary Integrity (Math vs. Prose)
- **Variable Encapsulation**: Identify naked variables in prose that should be in math mode (e.g., "a" → "$a$"). Use the document's prevailing typesetting (e.g., `$\mathrm{{M}}$` vs `$M$`) to resolve inconsistencies.
- **Prose Bleed-In**: Identify non-mathematical words/punctuation incorrectly pulled into math mode.
- **Operator Displacement**: Identify LaTeX commands (starting with `\`) that have leaked out of math mode into raw text and wrap them appropriately.
- Metadata **Protection**: DO NOT flag HTML Attributes (e.g., notation="...") as typos. Your audit is strictly limited to the Inner Text Node (the math/prose sitting between the opening <span...> or <b...> and closing </span> or </b> tags).
- **Metadata Protection**: DO NOT flag Attributes (e.g., notation="...", style="...") as typos. Your audit is strictly limited to the Inner Text (the text node residing between the opening and closing tag delimiters). Suggesting corrections that wrap existing text in new HTML tags is strictly forbidden.
- **Scope Restriction**: When reporting a typo that exists within a tagged element, the original and correction fields in your JSON must contain only the affected Inner Text. Do NOT include the <span...> or </span> delimiters themselves in the JSON fields.
Principle of Contentious Boundaries:
- **Identify Naked Variables**: Raw alphanumeric characters representing math objects (e.g., "field k", "characteristic p") MUST be wrapped in $k$ and $p$.
- **Respect Existing Enclosures**: If the text is already inside a LaTeX environment (e.g., between $$...$$, \begin{{aligned}}...\end{{aligned}}, or $...$), do NOT suggest adding additional delimiters.
- **The "Inner-Only" Rule**: Your audit applies only to the content between or outside opening and closing HTML tags. If the source is "<span>$\chi$ belongs to $\Delta(A)$</span>", the LaTeX is already valid as "Inner Text" for a math-renderer. Do not add $ unless the characters are literally sitting in a raw prose sentence without any math-mode context.

### II. Principles of Contextual Syntax (OCR Noise Reduction)
- **Symbolic Restoration**: Recognize visually similar misreads (e.g., `l` for `!`; `2` for `[`).
- **Structural Completion**: Fix "dangling" suffixes (e.g., `L-2d` → `$L[-2d]$`).
- **Command Repair**: Rejoin split LaTeX commands (e.g., `\operator name` → `\operatorname`).

### III. Principles of Structure (Translation & Lexical Mapping)
- **Lexical Recovery**: Replace literal translations with standard English mathematical terms (e.g., "Sheet" → "Sheaf", "Demonstration" → "Proof").

### IV. Hard Constraints (Formatting Immunity)
Strictly **PRESERVE** the following. Do not modify these even if they appear to contain typos:
- **Obsidian/Wiki Links**: `[[Note Name]]` or `[[Note Name|Alias]]`.
- **Footnotes**: `[^1]` and the corresponding `[^1]: Footnote text`.
- **Tag Delimiters**: Treat <span...>, </span>, <b...>, and </b> as Atomic Structural Markers. They are invisible to the audit. You are strictly prohibited from suggesting "corrections" that involve adding, removing, or "fixing" these delimiters. If the Inner Text is fine, skip it.
- **Markdown Markers**: Horizontal rules `---`, callouts `> [!INFO]`, and list markers.
- **Encoding Constraint**: You must output LaTeX backslashes as double-backslashes in the JSON (e.g., \\chi) to prevent string escaping errors. Never output literal tab (\t) or carriage return (\r) characters in the middle of a LaTeX command.
- **Filter Constraint**: If an Inner Text node contains no typos, do NOT include it in the JSON. The errors array should only contain actual proposed changes. Never include an entry where original and correction are identical.
- **Strict Exclusion**: If a segment of text contains no corruption, do NOT create an entry for it.
- **Deduplication**: Do not report the same typo multiple times for the same segment.
- **Actionable Only**: Every entry in errors must result in a string difference between original and correction.


### V. Output Requirement

Confidence Calibration:


    1.0: Certainty that the original text is a corruption (OCR noise, typo) and the correction is the only logical fix. For example, obvious OCR gore or standard term swaps or mistranslations ("Sheet" → "Sheaf").

    0.8 (Highly Likely): Naked variables in prose (e.g., "field k" → "field $k$") or missing or unbalanced braces.

    0.5 (Stylistic/Contextual): Changing \text to \mathrm or adding optional spacing for "clarity."

    0.1 (Heuristic/Suspicious): The text looks "garbled" or mathematically unusual, but a clear correction isn't certain. Use this for "best guesses" at heavily corrupted OCR lines where you might be hallucinating a fix.

Return ONLY a JSON object. Follow the structure of the schema below, but do not include the '$defs' or '$ref' keys in your response. 

WARNING: Do not include any HTML tags (<span>, <b>, etc.) in your correction strings unless the original text actually had a typo inside those tags. If the original text is $R$, your correction should not be <span>$R$</span>.

Critical Formatting Instruction:
All LaTeX commands in your JSON output MUST use double-backslashes (e.g., \\chi, \\rightarrow, \\operatorname). This ensures that the backslash is preserved as a literal character and not interpreted as a Python/JSON escape sequence (like \r or \t).

Required Object Structure:
{{
  "paper_title": "string",
  "errors": [
    {{
      "original": "string",
      "correction": "string",
      "reason": "string",
      "confidence": 0.0
    }}
  ]
}}

Full Schema for Reference:
{TypoReport.model_json_schema()}
"""


In [ ]:
#| export
def audit_text_for_typos(
        model_obj: SupportedLLM,
        text: str,
        context_text: Optional[str] = None,
        system_prompt: str = FIND_LATEX_TYPOS_SYSTEM_PROPT,
        config: Optional[dict] = None,
        max_context: int = 8192, 
        verbose: bool = True,
        message_to_print_before_audit: Optional[str] = None,
        message_to_print_upon_validation_error: Optional[str] = None,
    ) -> tuple[Optional[TypoReport], str]:
    """
    """
    # Use smart_truncate for context notes. 
    # We reserve 1/4 of the window for the target note and prompt to ensure they fit.
    reserved = max_context // 4 

    if context_text:
        context_text = smart_truncate(
            context_text, 
            model_obj, 
            max_context=max_context, 
            reserved_tokens=reserved, 
            verbose=verbose
        )
        
    user_msg = f"CONTEXT (Reference for notation/style):\n{context_text}\n\n---\nTARGET TEXT TO AUDIT:\n{text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg}
    ]

    # Set max_tokens to max_context to match your LM Studio environment settings.
    run_config = {
        "temperature": 0.0,
        "max_tokens": max_context, 
        **(config or {})
    }
    
    if verbose: 
        print(message_to_print_before_audit)
        print(messages)
        
    raw_response = call_llm(model_obj, messages, config=run_config, verbose=verbose)
    clean_response = process_llm_response(raw_response, return_thoughts=False)

    try:
        clean_json = clean_response.strip()
        if clean_json.startswith("```"):
            clean_json = re.sub(r'^```[a-z]*\n|```$', '', clean_json, flags=re.MULTILINE).strip()
        
        # JSON backslash fix for LaTeX
        clean_json = re.sub(r'\\(?![\\"/bfnrtu])', r'\\\\', clean_json)
        
        report = TypoReport.model_validate_json(clean_json)
        return report, raw_response
    except Exception:
        if verbose: 
            print(message_to_print_upon_validation_error)
        return None, raw_response


In [ ]:
#| export
def audit_note_content(
    model_obj: SupportedLLM,
    note: VaultNote,
    context_text: Optional[str] = None,
    system_prompt: str = FIND_LATEX_TYPOS_SYSTEM_PROPT,
    use_processed_note_content: bool = True,
    config: Optional[dict] = None,
    max_context: int = 8192, 
    verbose: bool = True
) -> tuple[Optional[TypoReport], str]:
    """
    Finds typos in a note, using max_context to define the model's total token limit.
    """
    if use_processed_note_content:
        note_content = str(process_standard_information_note(
            MarkdownFile.from_vault_note(note), note.vault))
    else:
        note_content = note.text()

    message_to_print_before_audit = f"  → Auditing [[{note.name}]] with max_tokens set to {max_context}\n"
    message_to_print_upon_validation_error = f"  ⚠️ Validation failed for [[{note.name}]]."
    return audit_text_for_typos(
        model_obj, note_content, context_text, system_prompt,
        config, max_context, verbose,
        message_to_print_before_audit=message_to_print_before_audit,
        message_to_print_upon_validation_error=message_to_print_upon_validation_error
        )
    
    # # Use smart_truncate for context notes. 
    # # We reserve 1/4 of the window for the target note and prompt to ensure they fit.
    # reserved = max_context // 4 

    # if context_text:
    #     context_text = smart_truncate(
    #         context_text, 
    #         model_obj, 
    #         max_context=max_context, 
    #         reserved_tokens=reserved, 
    #         verbose=verbose
    #     )
        
    # user_msg = f"CONTEXT (Reference for notation/style):\n{context_text}\n\n---\nTARGET TEXT TO AUDIT:\n{note_content}"
    # messages = [
    #     {"role": "system", "content": system_prompt},
    #     {"role": "user", "content": user_msg}
    # ]

    # # Set max_tokens to max_context to match your LM Studio environment settings.
    # run_config = {
    #     "temperature": 0.0,
    #     "max_tokens": max_context, 
    #     **(config or {})
    # }
    
    # if verbose: 
    #     print(f"  → Auditing [[{note.name}]] with max_tokens set to {max_context}")
    #     print(messages)
        
    # raw_response = call_llm(model_obj, messages, config=run_config, verbose=verbose)
    # clean_response = process_llm_response(raw_response, return_thoughts=False)

    # try:
    #     clean_json = clean_response.strip()
    #     if clean_json.startswith("```"):
    #         clean_json = re.sub(r'^```[a-z]*\n|```$', '', clean_json, flags=re.MULTILINE).strip()
        
    #     # JSON backslash fix for LaTeX
    #     clean_json = re.sub(r'\\(?![\\"/bfnrtu])', r'\\\\', clean_json)
        
    #     report = TypoReport.model_validate_json(clean_json)
    #     return report, raw_response
    # except Exception:
    #     if verbose: print(f"  ⚠️ Validation failed for [[{note.name}]].")
    #     return None, raw_response


In [ ]:
#| export
def typo_audit_log_message(
    report: Optional[TypoReport],
    raw_output: str,
    verbose: bool = True, 
    ) -> str:
    new_entry = ""
    if report:
        if not report.errors:
            new_entry += "*No typos found.*\n"
        for error in report.errors:
            is_safe = "[[" not in error.original or "[[" in error.correction
            safety = "" if is_safe else "⚠️ "
            new_entry += (
                f"- [ ] {safety}**Original**: `{error.original}`\n"
                f"  - **Correction**: `{error.correction}`\n"
                f"  - **Reason**: {error.reason} (Conf: {error.confidence})\n"
            )
    else:
        # If parsing failed, dump the raw response so it's not lost
        new_entry += "❌ **Parsing Error**: Could not validate JSON. Raw response below:\n\n"
        new_entry += f"```json\n{raw_output}\n```\n"
    return new_entry

In [ ]:
#| export
def log_audit_to_staging(
    staging_note: VaultNote, 
    target_note_name: str, 
    report: Optional[TypoReport],
    raw_output: str,
    verbose: bool = True
):
    """Writes results immediately. If report is None, writes the raw output."""
    new_entry = f"\n## [[{target_note_name}]]\n"
    
    new_entry += typo_audit_log_message(report, raw_output, verbose)
    # if report:
    #     if not report.errors:
    #         new_entry += "*No typos found.*\n"
    #     for error in report.errors:
    #         is_safe = "[[" not in error.original or "[[" in error.correction
    #         safety = "" if is_safe else "⚠️ "
    #         new_entry += (
    #             f"- [ ] {safety}**Original**: `{error.original}`\n"
    #             f"  - **Correction**: `{error.correction}`\n"
    #             f"  - **Reason**: {error.reason} (Conf: {error.confidence})\n"
    #         )
    # else:
    #     # If parsing failed, dump the raw response so it's not lost
    #     new_entry += "❌ **Parsing Error**: Could not validate JSON. Raw response below:\n\n"
    #     new_entry += f"```json\n{raw_output}\n```\n"

    # Persistent Write
    current_text = staging_note.text() if staging_note.exists() else "# Typo Audit Staging\n"
    staging_note.write(current_text + new_entry)


In [ ]:
#| export
def run_vault_audit(
    notes_to_audit: List[VaultNote],
    model_obj: SupportedLLM,
    staging_note_name: str = "_Typo_Staging_Review",
    context_window: int = 2,
    max_context: int = 8192,
    resume: bool = True,
    config: Optional[dict] = None,
    verbose: bool = True
):
    """
    Iterates through notes with strict resume logic and respects the provided max_context.
    """
    vault = notes_to_audit[0].vault
    staging_note = VaultNote(vault, rel_path=f"{staging_note_name}.md")
    
    if not staging_note.exists():
        staging_note.create()
        staging_note.write(f"# Typo Audit Staging\n*Generated on: {staging_note_name}*\n")
        existing_content = ""
    else:
        existing_content = staging_note.text()

    for i, note in enumerate(notes_to_audit):
        # Strict Resume Check
        note_header = f"## [[{note.name}]]"
        if resume and note_header in existing_content:
            following_text = existing_content.split(note_header)[-1].split("## [[")[0]
            if "❌ **Parsing Error**" not in following_text and "audit failed" not in following_text.lower():
                if verbose: print(f"[{i+1}/{len(notes_to_audit)}] ⏭️ Skipping [[{note.name}]]...")
                continue

        if verbose: print(f"[{i+1}/{len(notes_to_audit)}] 🔍 Processing [[{note.name}]]...")
        
        # Gather context
        start = max(0, i - context_window)
        end = min(len(notes_to_audit), i + context_window + 1)
        context_parts = []
        for j in range(start, end):
            if i == j: continue 
            ctx_note = notes_to_audit[j]
            try:
                info = str(process_standard_information_note(ctx_note, vault))
                context_parts.append(f"Related Note [[{ctx_note.name}]]:\n{info}")
            except:
                context_parts.append(f"Related Note [[{ctx_note.name}]]: (Fail)")
        
        context_text = "\n\n".join(context_parts)

        # Audit
        report, raw_output = audit_note_content(
            model_obj, 
            note, 
            context_text=context_text, 
            max_context=max_context,
            config=config,
            verbose=verbose
        )
        
        log_audit_to_staging(staging_note, note.name, report, raw_output, verbose=verbose)
        existing_content = staging_note.text()

    return staging_note

In [ ]:
#| hide
from fastcore.test import *

class MockRes:
    """Helper to simulate the LMS response object."""
    def __init__(self, content):
        self.content = content
        self.usage = {'completion_tokens': 10}

class MockModel:
    """A robust mock that handles both streaming and standard calls."""
    def __init__(self, responses: list):
        self.responses = responses
        self.index = 0
    
    def respond(self, chat, config=None, stream=False):
        if self.index >= len(self.responses):
            raise IndexError(f"MockModel exhausted: {len(self.responses)} responses provided.")
            
        content = self.responses[self.index]
        self.index += 1
        
        # If streaming is requested, return a list of 'chunks'
        if stream:
            # We simulate a single chunk containing the whole response
            return [MockRes(content)]
        
        # Otherwise return the single response object
        return MockRes(content)

# --- Updated Tests ---

# def test_run_math_proofreader():
#     # 1. Test Valid JSON
#     valid_json = '{"paper_title": "Test", "errors": []}'
#     mock = MockModel([valid_json])
#     report = run_math_proofreader(mock, "chunk", "prompt", verbose=False)
#     test_eq(report.paper_title, "Test")

#     # 2. Test Markdown Wrapping
#     wrapped = '```json\n{"paper_title": "Wrapped", "errors": []}\n```'
#     mock = MockModel([wrapped])
#     report = run_math_proofreader(mock, "chunk", "prompt", verbose=False)
#     test_eq(report.paper_title, "Wrapped")

#     # 3. Test Validation Error (Returns None)
#     mock = MockModel(['{"bad": "json"}'])
#     report = run_math_proofreader(mock, "chunk", "prompt", verbose=False)
#     test_eq(report, None)

# def test_retry_logic():
#     # We need 2 responses because the retry logic calls run_math_proofreader twice
#     responses = [
#         "Not JSON", # Pass 1 fails
#         '{"paper_title": "Fixed", "errors": []}' # Pass 2 succeeds
#     ]
#     mock = MockModel(responses)
#     report = run_math_proofreader_with_retry(mock, "chunk", "prompt", verbose=False)
#     test_eq(report.paper_title, "Fixed")
#     test_eq(mock.index, 2)

# test_run_math_proofreader()
# test_retry_logic()

## Applying typo fixes

In [ ]:
#| export
def batch_proofread_notes(
    notes: List[VaultNote], 
    model_obj: SupportedLLM, 
    staging_note_name: str = "Typo_Review_Staging",
    chunk_size: int = 3000,
    config: Optional[dict] = None,
    verbose: bool = True,
    resume: bool = True
) -> Optional[VaultNote]:
    if not notes: return None

    vault = notes[0].vault
    staging_note = VaultNote(vault, rel_path=f"{staging_note_name}.md")
    
    if not staging_note.exists():
        staging_note.create()
        staging_note.write(f"# 📝 Typo Review: {staging_note_name}\n\nCheck `[x]` to approve.\n\n---\n")
    